In [40]:
# 코드 실행에 필요한 라이브러리를 미리 import.
import csv
import os
import platform
import torch

import imageio.v2 as imageio
import numpy as np

from PIL import Image
from sklearn.model_selection import train_test_split

In [41]:
if platform.system() == 'Windows':
    HOME = 'C:/Users/first/'
else:
    HOME = '/home/ksy/'
PATH_PREFIX = f'{HOME}Develop/Kut-Deep-Learning-250204/'

# a_2d_image_data

In [42]:
# 이미지 읽기.
img_arr = imageio.imread(os.path.join(PATH_PREFIX, "_00_data", "a_image-dog", "bobby.jpg"))
print(type(img_arr))        # 이미지는 n차원 배열로 표현됨.
print(img_arr.shape)        # (rows, cols, channels)
print(img_arr.dtype)        # 각 필셀은 [0, 256) 범위의 값을 가짐 -> unit8

img = torch.from_numpy(img_arr)     # 넘파이 배열로부터 토치에서 사용할 수 있는 형태로 변화.
out = img.permute(2, 0, 1)          # (channels, rows, cols)
print(out.shape)

<class 'numpy.ndarray'>
(720, 1280, 3)
uint8
torch.Size([3, 720, 1280])


In [43]:
# 이미지 파일 이름 불러오기.
data_dir = os.path.join(PATH_PREFIX, "_00_data", "b_image-cats")
filenames = [
    name for name in os.listdir(data_dir) if os.path.splitext(name)[-1] == '.png'
]
print(filenames)

# 이미지 순회하기.
for i, filename in enumerate(filenames):
    image = Image.open(os.path.join(data_dir, filename))
    # image.show()  # 이미지 뷰어에서 열기.
    img_arr = imageio.imread(os.path.join(data_dir, filename))
    print(img_arr.shape)      # (rows, cols, channels)
    print(img_arr.dtype)      # uint8

# 넘파이 배열을 토치 텐서로 변환하기 위한 준비.
# 각 이미지는 하나의 batch 단위.
batch_size = 3
# (255, 255, 3)이 들어갈 공간이 세 개 필요하므로 (3, 256, 256, 3)
# 그런데 토치에서는 channels이 먼저 오므로 (3, 3, 256, 256)
batch = torch.zeros(batch_size, 3, 256, 256, dtype=torch.uint8)

# 이미지 순회하며,
for i, filename in enumerate(filenames):
    img_arr = imageio.imread(os.path.join(data_dir, filename))  # 이미지 읽고,
    img_t = torch.from_numpy(img_arr)                           # 넘파이 -> 토치 변환하고,
    img_t = img_t.permute(2, 0, 1)                              # channels가 먼저 오도록 하고,
    batch[i] = img_t                                            # batch로 복사.

# (3, 3, 256, 256)
print(batch.shape)

['cat2.png', 'cat1.png', 'cat3.png']
(256, 256, 3)
uint8
(256, 256, 3)
uint8
(256, 256, 3)
uint8
torch.Size([3, 3, 256, 256])


In [44]:
# 정규화.
batch = batch.float()   # uint8 -> float32
batch /= 255.0          # [0, 256) -> [0, 1)
print(batch.dtype)      # float32
print(batch.shape)      # (3, 3, 256, 256)

# RGB 포맷 파싱. (색상 채널이 몇 개인가)
n_channels = batch.shape[1]

# 채널 별로 순회.
for c in range(n_channels):
    mean = torch.mean(batch[:, c])      # 현재 색상의 전체 픽셀 평균.
    std = torch.std(batch[:, c])        # 현재 색상의 전체 픽셀 표준편차.
    print(mean, std)
    batch[:, c] = (batch[:, c] - mean) / std    # 평균: 0, 표준편차: 단위 표준편차.

torch.float32
torch.Size([3, 3, 256, 256])
tensor(0.5799) tensor(0.2212)
tensor(0.4493) tensor(0.2068)
tensor(0.3554) tensor(0.1931)


# b_tabular_wine_data_to_tensors

In [45]:
wine_path = os.path.join(PATH_PREFIX, "_00_data", "d_tabular-wine", "winequality-white.csv")
# CSV 형식 불러오기. 단, 구분자는 `;`.
# 첫 행 무시. 컬럼명.
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)
print(wineq_numpy.dtype)
print(wineq_numpy.shape)
print(wineq_numpy)
print()

# 컬럼명 가져오기.
# 파일의 내용을 불러오는 제너레이터로부터 첫 원소만 가져옴.
col_list = next(csv.reader(open(wine_path), delimiter=';'))
print(col_list)
print()

float32
(4898, 12)
[[ 7.    0.27  0.36 ...  0.45  8.8   6.  ]
 [ 6.3   0.3   0.34 ...  0.49  9.5   6.  ]
 [ 8.1   0.28  0.4  ...  0.44 10.1   6.  ]
 ...
 [ 6.5   0.24  0.19 ...  0.46  9.4   6.  ]
 [ 5.5   0.29  0.3  ...  0.38 12.8   7.  ]
 [ 6.    0.21  0.38 ...  0.32 11.8   6.  ]]

['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']



In [46]:
# 넘파이 -> 토치 형식 변환.
wineq = torch.from_numpy(wineq_numpy)
print(wineq.dtype)
print(wineq.shape)
print()

# 파이썬에서 `:-1`은 [begin, end - 1)을 의미.
# 즉, 마지막을 제외한 모든 열.
data = wineq[:, :-1]  # Selects all rows and all columns except the last
print(data.dtype)
print(data.shape)
print(data)
print()

# 파이썬에서 `-1`은 end - 1을 의미.
# 즉, 마지막을 열.
target = wineq[:, -1]  # Selects all rows and the last column
print(target.dtype)
print(target.shape)
print(target)
print()

# 자료형 변환: float32 -> int64
target = target.to(torch.int64)  # treat labels as an integer
print(target.dtype)
print(target.shape)
print(target)
print()

torch.float32
torch.Size([4898, 12])

torch.float32
torch.Size([4898, 11])
tensor([[ 7.0000,  0.2700,  0.3600,  ...,  3.0000,  0.4500,  8.8000],
        [ 6.3000,  0.3000,  0.3400,  ...,  3.3000,  0.4900,  9.5000],
        [ 8.1000,  0.2800,  0.4000,  ...,  3.2600,  0.4400, 10.1000],
        ...,
        [ 6.5000,  0.2400,  0.1900,  ...,  2.9900,  0.4600,  9.4000],
        [ 5.5000,  0.2900,  0.3000,  ...,  3.3400,  0.3800, 12.8000],
        [ 6.0000,  0.2100,  0.3800,  ...,  3.2600,  0.3200, 11.8000]])

torch.float32
torch.Size([4898])
tensor([6., 6., 6.,  ..., 6., 7., 6.])

torch.int64
torch.Size([4898])
tensor([6, 6, 6,  ..., 6, 7, 6])



In [47]:
# 10 * 10크기의 단위 행렬 생성.
# [1, 0, 0, ..., 0]
# [0, 1, 0, ..., 0]
# [0, 0, 1, ..., 0]
# ...
# [0, 0, 0, ..., 1]
eye_matrix = torch.eye(10)
# One-Hot Encoding: 원하는 열에 1, 원하는 열을 제외한 나머지 열에 0을 부여하는 방식.
# e.g.
#     데이터가 [a, b, c, d]일 때,
#     onehot을 [0, 1, 0, 0]으로 설정하면,
#     원하는 데이터는 `b`임을 알 수 있음.
# We use the 'target' tensor as indices to extract the corresponding rows from the identity matrix
# It can generate the one-hot vectors for each element in the 'target' tensor
# target이 n이면,[6, 6, 6,  ..., 6, 7, 6]
# [0, 0, 0, ..., 1, 0, 0, ..., 0]으로 변환.
#  ^^^^^^^^^^^^^
#  0이 n - 1개
onehot_target = eye_matrix[target]

# 데이터의 개수가 4989인 데이터에서 각각 길이가 10인 onehot 벡터를 추출하므로,
# 크기는 (4898, 10)이 된다.
# target: [6, 6, 6,  ..., 6, 7, 6]
# => [
#        [0, 0, 0, 0, 0, 0, 1, 0, ...],
#        [0, 0, 0, 0, 0, 0, 1, 0, ...],
#        [0, 0, 0, 0, 0, 0, 1, 0, ...],
#    ...
#        [0, 0, 0, 0, 0, 0, 1, 0, ...],
#        [0, 0, 0, 0, 0, 0, 0, 1, ...],
#        [0, 0, 0, 0, 0, 0, 1, 0, ...]
#    ]
print(onehot_target.shape)  # >>> torch.Size([4898, 10])
print(onehot_target[0])
print(onehot_target[1])
print(onehot_target[-2])
print(onehot_target)

torch.Size([4898, 10])
tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])
tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])
tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])


In [48]:
data_mean = torch.mean(data, dim=0)     # 각 데이터의 평균과,
data_var = torch.var(data, dim=0)       # 분산을 구해서,
data = (data - data_mean) / torch.sqrt(data_var)    # 데이터 표준화.
print(data)

tensor([[ 1.7208e-01, -8.1761e-02,  2.1326e-01,  ..., -1.2468e+00,
         -3.4915e-01, -1.3930e+00],
        [-6.5743e-01,  2.1587e-01,  4.7996e-02,  ...,  7.3995e-01,
          1.3422e-03, -8.2419e-01],
        [ 1.4756e+00,  1.7450e-02,  5.4378e-01,  ...,  4.7505e-01,
         -4.3677e-01, -3.3663e-01],
        ...,
        [-4.2043e-01, -3.7940e-01, -1.1915e+00,  ..., -1.3130e+00,
         -2.6153e-01, -9.0545e-01],
        [-1.6054e+00,  1.1666e-01, -2.8253e-01,  ...,  1.0049e+00,
         -9.6251e-01,  1.8574e+00],
        [-1.0129e+00, -6.7703e-01,  3.7852e-01,  ...,  4.7505e-01,
         -1.4882e+00,  1.0448e+00]])


In [49]:
# 주어진 데이터를 0.8 : 0.2로 나누어 학습 셋과 테스트 셋으로 나눔.
X_train, X_test, y_train, y_test = train_test_split(data, onehot_target, test_size=0.2)

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)

torch.Size([3918, 11])
torch.Size([3918, 10])
torch.Size([980, 11])
torch.Size([980, 10])


In [50]:
# 위의 과정을 하나의 함수로 분리.
def get_wine_data():
    # 파일 불러오기.
    wine_path = os.path.join(PATH_PREFIX, "_00_data", "d_tabular-wine", "winequality-white.csv")
    wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=";", skiprows=1)

    # 토치 형식으로 변환하기.
    wineq = torch.from_numpy(wineq_numpy)

    # 입력과 출력 분리하기.
    data = wineq[:, :-1]  # Selects all rows and all columns except the last
    target = wineq[:, -1].to(torch.int64)  # treat labels as an integer

    # onehot 변환.
    eye_matrix = torch.eye(10)
    onehot_target = eye_matrix[target]

    # 데이터 표준화.
    data_mean = torch.mean(data, dim=0)
    data_var = torch.var(data, dim=0)
    data = (data - data_mean) / torch.sqrt(data_var)

    # 학습 셋과 테스트 셋 선별.
    X_train, X_valid, y_train, y_valid = train_test_split(data, onehot_target, test_size=0.2)

    return X_train, X_valid, y_train, y_valid

print(get_wine_data())

(tensor([[ 0.5276, -1.0739,  0.2133,  ..., -0.3196, -0.6996,  1.5323],
        [-2.4350,  3.6883, -1.7699,  ...,  0.8724, -1.1378,  2.4262],
        [ 1.0016,  0.3151,  0.2133,  ..., -0.5183, -0.0863,  1.2073],
        ...,
        [ 2.0681,  0.9104,  3.0227,  ..., -1.2468,  2.1043, -0.9867],
        [ 0.1721, -0.3794, -0.7783,  ...,  1.8658, -0.2615, -0.2554],
        [ 1.1201,  0.2159,  0.5438,  ..., -0.3196, -0.8749,  0.3134]]), tensor([[ 1.9496,  2.7954,  3.3532,  ..., -1.3130,  1.8414, -1.0680],
        [ 1.3571,  2.6962, -1.3567,  ..., -0.6508, -1.6635, -0.8242],
        [-0.6574, -0.7762, -0.2825,  ...,  1.8658, -0.7873, -1.6368],
        ...,
        [ 0.6461,  0.3151, -0.6131,  ...,  0.4751, -1.1378, -1.1492],
        [-0.5389,  0.2159,  0.3785,  ..., -0.5845, -1.0501, -1.2305],
        [ 0.0536, -0.8755,  0.4612,  ..., -0.5183,  1.4909,  0.2322]]), tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        .